<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Random_Forest_for_Code_Switch_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import re

# ---------------------
# 1. Load ASCEND dataset (text-only)
# ---------------------
dataset = load_dataset("CAiRE/ASCEND", split="train")

# Remove audio-related columns
audio_columns = ["audio", "path", "duration", "original_speaker_id", "session_id", "topic", "language"]
dataset = dataset.remove_columns([c for c in dataset.column_names if c in audio_columns])

# Extract transcriptions
texts = [row["transcription"] for row in dataset if isinstance(row["transcription"], str)]
print(f"Loaded {len(texts)} transcriptions.")

# ---------------------
# 2. Tokenization & label generation
# ---------------------
def tokenize_and_label(text):
    tokens = re.findall(r'\b\w+\b|[\u4e00-\u9fff]', text)
    labels = []

    for i in range(len(tokens) - 1):
        next_token = tokens[i + 1]
        labels.append("zh" if re.search(r'[\u4e00-\u9fff]', next_token) else "en")

    return tokens[:-1], labels  # features = tokens, labels = next token lang

# Prepare dataset
data_tokens, data_labels = [], []
for text in texts[:500]:  # limit for demo
    tokens, labels = tokenize_and_label(text)
    data_tokens.extend(tokens)
    data_labels.extend(labels)

print(f"Prepared {len(data_tokens)} token samples.")

# ---------------------
# 3. Vectorize tokens
# ---------------------
vectorizer = CountVectorizer(analyzer="char", ngram_range=(1, 3))
X = vectorizer.fit_transform(data_tokens)
y = [1 if label == "zh" else 0 for label in data_labels]  # 1 = zh, 0 = en

# ---------------------
# 4. Train-test split
# ---------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------
# 5. Random Forest Classifier
# ---------------------
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# ---------------------
# 6. Evaluation
# ---------------------
print(classification_report(y_test, y_pred, target_names=["en (no_switch)", "zh (switch)"]))


Loaded 9869 transcriptions.
Prepared 971 token samples.
                precision    recall  f1-score   support

en (no_switch)       0.94      0.99      0.97       184
   zh (switch)       0.00      0.00      0.00        11

      accuracy                           0.94       195
     macro avg       0.47      0.50      0.48       195
  weighted avg       0.89      0.94      0.91       195

